# M5a — Convert the QuietNote fine-tune to LiteRT for MediaPipe

Converts `Sharangp/quietnote-m3-gemma4-e2b-merged` (the merged fp16 QLoRA
checkpoint from the M3 notebook) into a LiteRT-LM bundle so the MediaPipe
backend can run the fine-tune **in the app** via the dev-only model override
(`quietnote-model-url-override`, dev builds only — shipped with this notebook's PR).

**Who runs this:** Sharang, on Colab (standing M3 rule — the loop writes,
Sharang executes). Runtime: **High-RAM CPU is enough** (the exporter is
CPU-only; the fp16 checkpoint is ~9.6 GB, so a standard-RAM runtime will OOM).

## Honest tooling status (researched 2026-07-19 — read before running)

- The tool is **`litert-torch`** (the renamed successor of `ai-edge-torch` —
  the M5a task text predates the rename). Google's official Gemma 4 LiteRT-LM
  page documents `litert-torch export_hf` for `google/gemma-4-E2B-it`, which is
  exactly our base — but only via the **nightly** package, and it emits a
  **`.litertlm`** bundle, not the `.task` the app currently loads.
- Known open upstream bugs: exporting with the model-type override can still
  stamp **generic metadata** instead of gemma4 ([litert-torch#1001](https://github.com/google-ai-edge/litert-torch/issues/1001));
  the Conversation API can fail on the chat template without the jinja
  override ([LiteRT-LM#2078](https://github.com/google-ai-edge/LiteRT-LM/issues/2078));
  [litert-torch#998](https://github.com/google-ai-edge/litert-torch/issues/998)
  (the exact "fine-tuned gemma-4 safetensors → LiteRT-LM" question) is unanswered.
- The recipe behind the official **web** artifact
  (`litert-community/gemma-4-E2B-it-litert-lm/gemma-4-E2B-it-web.task`) is
  **unpublished** — same situation as the ONNX conversion (M4a section).
  Whether `@mediapipe/tasks-genai@0.10.27` in the browser accepts a gemma4
  `.litertlm` is **the experiment the dev override exists to answer** (it does
  accept `.litertlm` for gemma-3n models per Google's own web docs).
- If the `.litertlm` fails to load in-app, the fallback ladder is at the bottom
  of this notebook. Do not burn hours fighting the exporter — record what
  happened in `docs/initiatives/model-quality.md` and stop.

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────
MERGED_REPO = "Sharangp/quietnote-m3-gemma4-e2b-merged"  # fp16 merged checkpoint (private)
OUT_REPO = "Sharangp/quietnote-m3-gemma4-e2b-litert"      # where the bundle gets pushed (private)
LOCAL_CKPT = "/content/merged"
OUT_DIR = "/content/litert-out"

from getpass import getpass
import os
os.environ["HF_TOKEN"] = getpass("Paste the Sharangp HF write token (never committed anywhere): ")

In [ ]:
# ── Install the exporter (nightly — gemma4 support lives there) ─────────
%pip install -q --upgrade litert-torch-nightly huggingface_hub
!litert-torch export_hf --help | head -60  # sanity: the CLI exists; skim the real flag names

In [ ]:
# ── Download the merged checkpoint (~9.6 GB) ────────────────────────────
from huggingface_hub import snapshot_download
snapshot_download(MERGED_REPO, local_dir=LOCAL_CKPT, token=os.environ["HF_TOKEN"])
!ls -lh {LOCAL_CKPT}

In [ ]:
# ── Export: fine-tuned safetensors → .litertlm ──────────────────────────
# Flags per Google's Gemma 4 LiteRT-LM page (developers.google.com/edge/litert-lm/models/gemma-4):
#   --externalize_embedder            (their documented gemma-4 invocation)
#   --jinja_chat_template_override    (pulls the official chat template — avoids LiteRT-LM#2078)
# Expect this to take a while on CPU (the 1B-class docs quote 10–30 min; E2B is bigger).
!litert-torch export_hf \
  --model={LOCAL_CKPT} \
  --output_dir={OUT_DIR} \
  --externalize_embedder \
  --jinja_chat_template_override=litert-community/gemma-4-E2B-it-litert-lm
!ls -lh {OUT_DIR}

In [ ]:
# ── FALLBACK (only if the CLI export fails or stamps generic metadata) ──
# The Python API exposes the gemma4 model-type override the CLI may miss
# (litert-torch#1001 — verify the bundle metadata says gemma4, not generic).
# Uncomment and run:
#
# from litert_torch.generative.utilities import hf_export
# hf_export.export(
#     model=LOCAL_CKPT,
#     output_dir=OUT_DIR,
#     externalize_embedder=True,
#     bundle_litert_lm=True,
#     litert_lm_model_type_override="gemma4",
# )
# NOTE: exact signature may drift on nightly — `help(hf_export.export)` first.

In [ ]:
# ── Push the bundle to HF (private, Sharangp) ───────────────────────────
from huggingface_hub import HfApi
api = HfApi(token=os.environ["HF_TOKEN"])
api.create_repo(OUT_REPO, private=True, exist_ok=True)
api.upload_folder(folder_path=OUT_DIR, repo_id=OUT_REPO)
print(f"Pushed. Download the .litertlm from https://huggingface.co/{OUT_REPO}")

## In-app test (back on the dev machine — the actual M5a verification)

1. Download the exported bundle next to the repo (NOT inside it — multi-GB):
   `hf download Sharangp/quietnote-m3-gemma4-e2b-litert --local-dir C:\\Users\\shara\\m5a-work`
2. Serve it with CORS (the app runs on :5173, the file on :8080):
   `npx http-server C:\\Users\\shara\\m5a-work -p 8080 --cors`
3. `npm run dev`, open http://localhost:5173, DevTools console:
   ```js
   localStorage.setItem("quietnote-model-url-override", "http://localhost:8080/<file>.litertlm");
   localStorage.setItem("quietnote-runtime", "mediapipe");
   location.reload();
   ```
   The console must show `[quietnote] DEV model override active — …` and the
   network tab a fetch to localhost:8080 (cached in `mediapipe-cache` under the
   override URL, so re-runs are instant).
4. **Success** = a full journal exchange streams from the fine-tune (its voice is
   unmistakable vs base — see the M4a transcripts). Screenshot →
   `docs/screenshots/`, note the result in `docs/initiatives/model-quality.md`.
5. Cleanup: `localStorage.removeItem("quietnote-model-url-override")` and delete
   the `mediapipe-cache` entry in DevTools → Application → Cache Storage.

**If `LlmInference.createFromOptions` rejects the bundle** (web runtime can't
read a gemma4 `.litertlm`): (a) try the newest `@mediapipe/tasks-genai` in a
scratch branch (pin rule: bump BOTH package.json and `TASKS_GENAI_VERSION`);
(b) ask in the `litert-community/gemma-4-E2B-it-litert-lm` HF discussions how
`gemma-4-E2B-it-web.task` was produced (same move as the ONNX recipe question);
(c) record the block in the initiative doc — do not force it.

**Safety reminder:** this model FAILED the release-gate medical_refusal floors
(M4a, 2026-07-18). The override is a dev-only pipeline test — nothing here
ships, and M5 proper waits for the full-data retrain to clear ALL floors.